In [ ]:
#@title 1. Setup Environment
import os, sys, json, time
from google.colab import files
import numpy as np
import torch

# Clone ProteinMPNN
if not os.path.isdir("ProteinMPNN"):
    os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")
sys.path.append('/content/ProteinMPNN')

print("✅ ProteinMPNN cloned and ready")

✅ ProteinMPNN cloned and ready


In [ ]:
#@title 2. Load ProteinMPNN Model
import warnings
warnings.filterwarnings("ignore")

from protein_mpnn_utils import parse_PDB, ProteinMPNN, tied_featurize

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model settings - good balance of quality/speed
model_name = "v_48_020"  # @param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]
backbone_noise = 0.00

path_to_model_weights = '/content/ProteinMPNN/vanilla_model_weights'
checkpoint_path = f"{path_to_model_weights}/{model_name}.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)
model = ProteinMPNN(num_letters=21,
                    node_features=128,
                    edge_features=128,
                    hidden_dim=128,
                    num_encoder_layers=3,
                    num_decoder_layers=3,
                    augment_eps=backbone_noise,
                    k_neighbors=checkpoint['num_edges'])
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Model {model_name} loaded (noise level: {checkpoint['noise_level']}Å)")

Using device: cuda:0
✅ Model v_48_020 loaded (noise level: 0.2Å)


In [ ]:
#@title 3. Toxin Seed List + Helpers
import re
import numpy as np
from google.colab import files
from protein_mpnn_utils import parse_PDB, StructureDatasetPDB, _S_to_seq

#old_toxin_list = [
#    {"name": "Alpha-Bungarotoxin", "pdb": "1IK8", "chain": "A"},
#    {"name": "Charybdotoxin", "pdb": "2CRD", "chain": "A"},
#    {"name": "Agitoxin-2", "pdb": "1AGT", "chain": "A"},
#    {"name": "Kappa-Bungarotoxin", "pdb": "1KBA", "chain": "A"},
#]

toxin_list = [
    {"name": "Influenza A virus", "pdb": "8F38", "chain": "A"},
    {"name": "Influenza A virus", "pdb": "9V44", "chain": "A"},
    {"name": "Influenza A virus", "pdb": "9GSP", "chain": "A"},
    {"name": "SARs-CoV-2 virus", "pdb": "8YAX", "chain": "A"},
    {"name": "Zika virus", "pdb": "5TFR", "chain": "A"},
    {"name": "Zika virus", "pdb": "5M2X", "chain": "A"}, #
    {"name": "Sindbis virus", "pdb": "3MUU", "chain": "A"},
    {"name": "Vaccinia virus", "pdb": "7SEP", "chain": "A"},
    {"name": "Vaccinia virus", "pdb": "5JK7", "chain": "A"}, # also: ['Q16531', 'Q9Y4B6', 'P13051', 'P12520']
    {"name": "Vaccinia virus", "pdb": "3MI9", "chain": "A"}, # also: ['P50750', 'O60563', 'P04608']
    # {"name": "Measles virus", "uniprot": "P50750", "pdb": "3MI9"} for Measles virus there are no able nucleotides cores. + chain H in pdb

    # example
    # Add more if you want
]

def get_pdb(pdb_code):
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    return f"{pdb_code}.pdb"

def compute_sequence_identity(seq1, seq2):
    matches = sum(a == b for a, b in zip(seq1, seq2))
    return (matches / max(len(seq1), len(seq2))) * 100 if max(len(seq1), len(seq2)) > 0 else 0

In [ ]:
#@title 4. 🚀 Improved Red-Teaming Loop (Better Output + Filtering)

# ================== CONFIG ==================
pdb_code = "1IK8"          # Change this
designed_chains = "A"
num_seq_per_target = 16    # Increase if you want
sampling_temp = 0.25
min_identity_threshold = 40  # Only save if below this
# ===========================================

pdb_path = get_pdb(pdb_code)
designed_chain_list = re.sub("[^A-Za-z]+", ",", designed_chains).split(",")
fixed_chain_list = []

chain_list = list(set(designed_chain_list + fixed_chain_list))

pdb_dict_list = parse_PDB(pdb_path, input_chain_list=chain_list)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)

chain_id_dict = {pdb_dict_list[0]['name']: (designed_chain_list, fixed_chain_list)}

print(f"🎯 Designing {pdb_code} ...")

fixed_positions_dict = omit_AA_dict = tied_positions_dict = pssm_dict = bias_by_res_dict = None
omit_AAs_np = np.array([False] * 21)
bias_AAs_np = np.zeros(21)

saved_count = 0

with torch.no_grad():
    for protein in dataset_valid:
        batch_clones = [protein] * 1

        X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, \
        visible_list_list, masked_list_list, masked_chain_length_list_list, \
        chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, \
        pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = \
            tied_featurize(batch_clones, device, chain_id_dict,
                          fixed_positions_dict, omit_AA_dict, tied_positions_dict,
                          pssm_dict, bias_by_res_dict)

        for temp in [sampling_temp]:
            print(f"\n=== Temperature = {temp} ===")
            for i in range(num_seq_per_target):
                randn = torch.randn(chain_M.shape, device=X.device)

                sample_dict = model.sample(
                    X, randn, S, chain_M, chain_encoding_all, residue_idx, mask=mask,
                    temperature=temp,
                    omit_AAs_np=omit_AAs_np,
                    bias_AAs_np=bias_AAs_np,
                    chain_M_pos=chain_M_pos,
                    omit_AA_mask=omit_AA_mask,
                    pssm_coef=pssm_coef,
                    pssm_bias=pssm_bias,
                    pssm_multi=0.0,
                    pssm_log_odds_flag=False,
                    pssm_log_odds_mask=None,
                    pssm_bias_flag=False,
                    bias_by_res=bias_by_res_all
                )

                S_sample = sample_dict["S"][0].cpu().numpy()
                seq_str = _S_to_seq(S_sample, chain_M[0])

                original_seq = pdb_dict_list[0][f"seq_chain_{designed_chains}"]
                identity = compute_sequence_identity(original_seq, seq_str)

                print(f"Variant {i+1:02d} | Identity: {identity:.1f}%")

                if identity < min_identity_threshold:
                    with open("designed_toxins.fasta", "a") as f:
                        f.write(f">redesign_{pdb_code}_T{temp}_id{identity:.1f}_var{i}\n{seq_str}\n")
                    saved_count += 1

print(f"\n✅ Finished! Saved {saved_count} low-identity sequences to designed_toxins.fasta")
files.download("designed_toxins.fasta")

🎯 Designing 1IK8 ...

=== Temperature = 0.25 ===
Variant 01 | Identity: 21.6%
Variant 02 | Identity: 18.9%
Variant 03 | Identity: 21.6%
Variant 04 | Identity: 20.3%
Variant 05 | Identity: 16.2%
Variant 06 | Identity: 23.0%
Variant 07 | Identity: 17.6%
Variant 08 | Identity: 17.6%
Variant 09 | Identity: 17.6%
Variant 10 | Identity: 16.2%
Variant 11 | Identity: 21.6%
Variant 12 | Identity: 21.6%
Variant 13 | Identity: 16.2%
Variant 14 | Identity: 17.6%
Variant 15 | Identity: 24.3%
Variant 16 | Identity: 18.9%

✅ Finished! Saved 16 low-identity sequences to designed_toxins.fasta


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title 5. 🚀 RANDOM 15-TOXIN BATCH (Fixed - Uses Known PDBs + Fallback)

!pip install -q biopython

import random
from Bio import SeqIO
import numpy as np
from google.colab import files

# ================== CONFIG ==================
num_toxins = 15
num_seq_per_target = 8
sampling_temp = 0.08
min_identity_threshold = 40
random_seed = 42
# ===========================================

print("🎲 Building batch with toxins that have PDB structures...")

# Expanded list of toxins with reliable PDBs (short, good for Colab)
#old_pdb_toxins = [
#    {"name": "Alpha-Bungarotoxin", "uniprot": "P60615", "pdb": "1IK8"},
#    {"name": "Charybdotoxin",      "uniprot": "P13487", "pdb": "2CRD"},
#    {"name": "Agitoxin-2",         "uniprot": "P46111", "pdb": "1AGT"},
#    {"name": "Kappa-Bungarotoxin", "uniprot": "P01398", "pdb": "1KBA"},
#    {"name": "Ricin A chain",      "uniprot": "P02879", "pdb": "2AAI"},
#    {"name": "Conotoxin",          "uniprot": "P0C1X2", "pdb": "1TTG"},   # example
#    # Add more if you want
#]

known_pdb_toxins = [
    {"name": "Influenza A virus", "uniprot": "Q289M7", "pdb": "8F38"},
    {"name": "Influenza A virus", "uniprot": "P31345", "pdb": "9V44"},
    {"name": "Influenza A virus", "uniprot": "A0A6M4YP75", "pdb": "9GSP"},
    {"name": "SARs-CoV-2 virus", "uniprot": "P0DTD1", "pdb": "8YAX"},
    {"name": "Zika virus", "uniprot": "Q32ZE1", "pdb": "5TFR"},
    {"name": "Zika virus", "uniprot": "A0A0X8GJ44", "pdb": "5M2X"},#
    {"name": "Sindbis virus", "uniprot": "P03316", "pdb": "3MUU"},
    {"name": "Vaccinia virus", "uniprot": "P03366", "pdb": "7SEP"},
    {"name": "Vaccinia virus", "uniprot": "P13051", "pdb": "5JK7"}, # also: ['Q16531', 'Q9Y4B6', 'P13051', 'P12520']
    {"name": "Vaccinia virus", "uniprot": "P50750", "pdb": "3MI9"} # also: ['P50750', 'O60563', 'P04608']
    # {"name": "Measles virus", "uniprot": "P50750", "pdb": "3MI9"} for Measles virus there are no able nucleotides cores. + chain H in pdb

    # example
    # Add more if you want
]



# If we need more, we can fall back to random with PDBs, but for speed we use these + random from a small set
random.seed(random_seed)
selected = random.choices(known_pdb_toxins, k=num_toxins)   # with replacement is fine for testing

print(f"Selected {len(selected)} toxins with PDBs\n")

toxin_list = selected   # already has pdb

# ====================== Generation ======================
saved_total = 0
open("critical_9_toxin_batch.fasta", "w").close()  # create empty file

with torch.no_grad():
    for toxin in toxin_list:
        print(f"\n🎯 Processing {toxin['name']} ({toxin['pdb']})")
        pdb_path = get_pdb(toxin['pdb'])

        designed_chain_list = ["A"]
        fixed_chain_list = []

        pdb_dict_list = parse_PDB(pdb_path, input_chain_list=designed_chain_list)
        dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)

        chain_id_dict = {pdb_dict_list[0]['name']: (designed_chain_list, fixed_chain_list)}

        fixed_positions_dict = omit_AA_dict = tied_positions_dict = pssm_dict = bias_by_res_dict = None
        omit_AAs_np = np.array([False] * 21)
        bias_AAs_np = np.zeros(21)

        toxin_saved = 0

        for protein in dataset_valid:
            batch_clones = [protein] * 1

            X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, \
            visible_list_list, masked_list_list, masked_chain_length_list_list, \
            chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, \
            pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = \
                tied_featurize(batch_clones, device, chain_id_dict,
                              fixed_positions_dict, omit_AA_dict, tied_positions_dict,
                              pssm_dict, bias_by_res_dict)

            for i in range(num_seq_per_target):
                randn = torch.randn(chain_M.shape, device=X.device)

                sample_dict = model.sample(
                    X, randn, S, chain_M, chain_encoding_all, residue_idx, mask=mask,
                    temperature=sampling_temp,
                    omit_AAs_np=omit_AAs_np,
                    bias_AAs_np=bias_AAs_np,
                    chain_M_pos=chain_M_pos,
                    omit_AA_mask=omit_AA_mask,
                    pssm_coef=pssm_coef,
                    pssm_bias=pssm_bias,
                    pssm_multi=0.0,
                    pssm_log_odds_flag=False,
                    pssm_log_odds_mask=None,
                    pssm_bias_flag=False,
                    bias_by_res=bias_by_res_all
                )

                S_sample = sample_dict["S"][0].cpu().numpy()
                seq_str = _S_to_seq(S_sample, chain_M[0])

                original_seq = pdb_dict_list[0][f"seq_chain_A"]
                identity = compute_sequence_identity(original_seq, seq_str)

                if identity < min_identity_threshold:
                    with open("random_15_toxin_batch.fasta", "a") as f:
                        header = f">redesign_{toxin['name']}_{toxin['pdb']}_T{sampling_temp}_id{identity:.1f}_var{i}"
                        f.write(f"{header}\n{seq_str}\n")
                    toxin_saved += 1
                    saved_total += 1

                print(f"   Variant {i+1:02d} | Identity: {identity:.1f}%")

        print(f"   → Saved {toxin_saved} variants\n")

print(f"\n🎉 BATCH COMPLETE! Total sequences generated: {saved_total}")
files.download("critical_9_toxin_batch.fasta")

🎲 Building batch with toxins that have PDB structures...
Selected 15 toxins with PDBs


🎯 Processing Sindbis virus (3MUU)
   Variant 01 | Identity: 32.1%
   Variant 02 | Identity: 33.5%
   Variant 03 | Identity: 32.5%
   Variant 04 | Identity: 34.1%
   Variant 05 | Identity: 35.0%
   Variant 06 | Identity: 32.5%
   Variant 07 | Identity: 33.1%
   Variant 08 | Identity: 32.0%
   → Saved 8 variants


🎯 Processing Influenza A virus (8F38)
   Variant 01 | Identity: 40.5%
   Variant 02 | Identity: 44.4%
   Variant 03 | Identity: 39.9%
   Variant 04 | Identity: 42.9%
   Variant 05 | Identity: 43.6%
   Variant 06 | Identity: 41.7%
   Variant 07 | Identity: 40.7%
   Variant 08 | Identity: 40.5%
   → Saved 1 variants


🎯 Processing Influenza A virus (9GSP)
   Variant 01 | Identity: 37.3%
   Variant 02 | Identity: 34.7%
   Variant 03 | Identity: 36.7%
   Variant 04 | Identity: 36.3%
   Variant 05 | Identity: 36.3%
   Variant 06 | Identity: 35.9%
   Variant 07 | Identity: 34.9%
   Variant 08 | Id

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title 6. 🔍 Quick Check on the Latest Generated Batch

!pip install -q biopython

from Bio import SeqIO
from google.colab import files

# Use the correct filename from the new Cell 5
input_fasta = "random_15_toxin_batch.fasta"   # ← Updated

print("=== First 10 Generated Sequences ===\n")
sequences = list(SeqIO.parse(input_fasta, "fasta"))

for i, record in enumerate(sequences[:10]):
    seq = str(record.seq)
    print(f"{record.id}")
    print(f"{seq[:80]}{'...' if len(seq)>80 else ''}")
    print(f"Length: {len(seq)} aa\n")

print(f"Total sequences in batch: {len(sequences)}")
print("\n✅ Ready for SecureDNA + ESM2 screening!")

=== First 10 Generated Sequences ===

redesign_Sindbis
KYTTKPYKGYCESSTTGTPAYTPFAIESVYDDADDGTVLIKTTALFGVGTDGSDDPNGFQKASATGDGTYESGSLSDVTV...
Length: 735 aa

redesign_Sindbis
TLTTKAYKGYCTSCTTGKPCYTPLAILDVDDLADDNTVLIKTTAIFGVGDTGDNNPNKARMASKTGDGTLESFDAKDIKV...
Length: 735 aa

redesign_Sindbis
KLTTKPYKGYCESSTDGKPAYTPLAIEDVYDVADDNTVLIKTTALFGVGENGDNNPNKMRYLSEDGDGKYESKDLSSVKV...
Length: 735 aa

redesign_Sindbis
KYTTKPFRGKCSSCTTGKPCDTPLAILDVEDVADDNQVLIKTTALFGVGKDGSNDPNKARMLPEDGSGKYESFSTDDIVV...
Length: 735 aa

redesign_Sindbis
TLTTKPYLGYCESCTDGKPCYSPLAIESVEDLADTNQVLIKTTALFGAGENGDNNPNKARMASPDGDGTYETYSISDVTV...
Length: 735 aa

redesign_Sindbis
TLTTKPFLGKCTSSTTGTPATTPLAILSVDDSADDNQVLIKTTAIFGAGETGGNDPNKYRMASTTGDGTYTSGSASDVKV...
Length: 735 aa

redesign_Sindbis
TLTTKPYRGYCESCTTGKPCYSPLAIEEVYDLADDNQVLIKTTALFGVGENGSNNPNSAAKASETGDGTYESFSTSSITV...
Length: 735 aa

redesign_Sindbis
TYTTKPYKGYCESCTDGKPCYTPLAILDVYDLADTNQVLIKTTALFGVGKNGDNNPNAAAMASKDGNGTIESVSLSDVTV...
Length: 735 aa

redesign_Influenza
GKIIVGY